# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayesha-Shahzadkhan/flyrank-assignment1/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [57]:
!git clone https://github.com/Ayesha-Shahzadkhan/flyrank-assignment1.git
%cd flyrank-assignment1

Cloning into 'flyrank-assignment1'...
remote: Enumerating objects: 153, done.
remote: Counting objects: 100% (153/153), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 153 (delta 62), reused 99 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (153/153), 1.87 MiB | 5.23 MiB/s, done.
Resolving deltas: 100% (62/62), done.
/content/flyrank-assignment1/flyrank-assignment1/flyrank-assignment1/flyrank-assignment1


In [58]:
import os
print(os.path.exists("data/raw/content_refresh_anonymized.csv"))

True


In [59]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
df.columns.tolist()

(30000, 44)


['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [60]:
df["trend_direction"].value_counts()

,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


In [61]:
df['days_since_last_update'].describe()

,days_since_last_update
count,30000.000000
mean,46.098300
std,42.078709
min,1.000000
25%,20.000000
50%,20.000000
75%,104.000000
max,373.000000


In [62]:
bins = [0, 20, 50, 104, 373]
labels = ["<=20", "21-50", "51-104", "105+"]
df['staleness_bucket'] = pd.cut(df['days_since_last_update'], bins=bins, labels=labels)
df['staleness_bucket'].value_counts()

,count
staleness_bucket,
<=20,15866
51-104,9080
21-50,4736
105+,318


In [63]:
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
df['is_declining'].value_counts()

summary = df.groupby('staleness_bucket')['is_declining'].agg(['count', 'mean'])
print(summary)

                  count      mean
staleness_bucket                 
<=20              15866  0.538888
21-50              4736  0.421030
51-104             9080  0.610573
105+                318  0.547170


/tmp/ipykernel_12971/726302028.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary = df.groupby('staleness_bucket')['is_declining'].agg(['count', 'mean'])


## Signal Check #1 — Staleness (`days_since_last_update`)

**Bucket table:**

| Bucket | n | declining-rate |
|---|---|---|
| <=20 | 15,866 | 0.539 |
| 21-50 | 4,736 | 0.421 |
| 51-104 | 9,080 | 0.611 |
| 105+ | 318 | 0.547 |

**Verdict: MIXED**

The assumption was that staler content carries higher decline risk (a monotonic increase across buckets). The actual pattern is non-monotonic — the 21-50 day bucket has the lowest decline rate (0.421), followed by a jump to the highest rate in 51-104 days (0.611), then a drop again at 105+ (0.547). No consistent one-directional trend emerged, so staleness alone is not a reliable decline predictor in this bucketed form.

Note: the 105+ bucket has only n=318 (much smaller than the other buckets), so its estimate is noisier and shouldn't be weighted heavily.

In [64]:
df['impressions_90d'].describe()

,impressions_90d
count,30000.000000
mean,5200.366300
std,16838.019547
min,1.000000
25%,81.000000
50%,731.000000
75%,3615.250000
max,517715.000000


In [65]:
bins = [0, 81, 731, 3615, float("inf")]
labels = ["<=81", "82-731", "732-3615", "3615+"]
df['impression_bucket'] = pd.cut(df['impressions_90d'], bins=bins, labels=labels)
df['impression_bucket'].value_counts()

,count
impression_bucket,
<=81,7503
3615+,7500
82-731,7499
732-3615,7498


In [66]:
summary2 = df.groupby('impression_bucket')['is_declining'].agg(['count', 'mean'])
print(summary2)

                   count      mean
impression_bucket                 
<=81                7503  0.376116
82-731              7499  0.604614
732-3615            7498  0.625634
3615+               7500  0.562000


/tmp/ipykernel_12971/366428798.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary2 = df.groupby('impression_bucket')['is_declining'].agg(['count', 'mean'])


## Signal Check #2 — Visibility (`impressions_90d`)

**Bucket table:**

| Bucket | n | declining-rate |
|---|---|---|
| <=81 | 7,503 | 0.376 |
| 82-731 | 7,499 | 0.605 |
| 732-3615 | 7,498 | 0.626 |
| 3615+ | 7,500 | 0.562 |

**Verdict: CONFIRMED**

Higher visibility (impressions) correlates with a meaningfully higher decline rate — the lowest-visibility bucket declines at 37.6%, compared to 56-63% for all higher-visibility buckets. The direction holds across most of the range (3 of 4 buckets increase), with only a mild dip at the very top bucket. Bucket sizes are balanced (~7,500 each, from quartile-based bins), so no small-sample caveat applies here.

## Rule (plain words)

A page is a priority for review if it has **high visibility (impressions)** — the confirmed driver of decline risk (Signal Check #2). Staleness alone did not show a reliable, consistent pattern (Signal Check #1: MIXED), so it is used only as a **secondary tiebreaker** among high-visibility pages, not as the main driver.

## Reason codes

| Visibility | Staleness | Reason code |
|---|---|---|
| High (impressions > 81) | Old (days_since_last_update >= 51) | `HIGH_VISIBILITY_STALE` |
| High (impressions > 81) | Fresh (days_since_last_update < 51) | `HIGH_VISIBILITY_FRESH` |
| Low (impressions <= 81) | — | `LOW_VISIBILITY` |

**Why these thresholds:** both cutoffs come directly from the bucket tables above — 81 impressions is where the decline rate jumps sharply (37.6% → 60.5%+), and 51 days is where the staleness bucket with the highest decline rate (51-104 days, 0.611) begins.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [67]:
df['score'] = df['impressions_90d']

high_visibility = df['impressions_90d'] > 81
staleness = df['days_since_last_update'] > 51

df['reason_code'] = 'LOW_VISIBILITY'
df.loc[high_visibility & staleness, 'reason_code'] = 'HIGH_VISIBILITY_STALE'
df.loc[high_visibility & -staleness, 'reason_code'] = 'HIGH_VISIBILITY_FRESH'

In [68]:
action_map = {
    'LOW_VISIBILITY' : 'NO ACTION',
    'HIGH_VISIBILITY_STALE' : 'REFRESH',
    'HIGH_VISIBILITY_FRESH' : 'REVIEW'
}
df['action'] = df['reason_code'].map(action_map)

In [69]:
df_ranked = df.sort_values('score', ascending=False)

import os
os.makedirs('work/outputs', exist_ok=True)
df_ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)


In [70]:
import pandas as pd
check = pd.read_csv("work/outputs/baseline_action_score.csv")
print(check.shape)

print(check["reason_code"].value_counts())

check[["content_id", "impressions_90d", "days_since_last_update", "score", "reason_code", "action"]].head()

(30000, 50)
reason_code
HIGH_VISIBILITY_FRESH    14251
HIGH_VISIBILITY_STALE     8246
LOW_VISIBILITY            7503
Name: count, dtype: int64


,content_id,impressions_90d,days_since_last_update,score,reason_code,action
0,content_5fe46e04994d,517715,104,517715,HIGH_VISIBILITY_STALE,REFRESH
1,content_aaef01a50def,517109,22,517109,HIGH_VISIBILITY_FRESH,REVIEW
2,content_8c19996aa890,509252,20,509252,HIGH_VISIBILITY_FRESH,REVIEW
3,content_2cb567c3c89b,497727,48,497727,HIGH_VISIBILITY_FRESH,REVIEW
4,content_4c36c775b818,463103,20,463103,HIGH_VISIBILITY_FRESH,REVIEW


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [71]:
top20 = check.head(20)[["content_id", "impressions_90d", "days_since_last_update", "score", "reason_code", "action"]]
top20

,content_id,impressions_90d,days_since_last_update,score,reason_code,action
0,content_5fe46e04994d,517715,104,517715,HIGH_VISIBILITY_STALE,REFRESH
1,content_aaef01a50def,517109,22,517109,HIGH_VISIBILITY_FRESH,REVIEW
2,content_8c19996aa890,509252,20,509252,HIGH_VISIBILITY_FRESH,REVIEW
3,content_2cb567c3c89b,497727,48,497727,HIGH_VISIBILITY_FRESH,REVIEW
4,content_4c36c775b818,463103,20,463103,HIGH_VISIBILITY_FRESH,REVIEW
5,content_2dba2b1f9536,443434,104,443434,HIGH_VISIBILITY_STALE,REFRESH
6,content_1a9e894be2e2,416180,22,416180,HIGH_VISIBILITY_FRESH,REVIEW
7,content_2c2606c5d176,347399,104,347399,HIGH_VISIBILITY_STALE,REFRESH
8,content_db5989a78dd3,345111,20,345111,HIGH_VISIBILITY_FRESH,REVIEW
9,content_44e481c8f55b,312694,20,312694,HIGH_VISIBILITY_FRESH,REVIEW


## Top-20 Review

**1. content_5fe46e04994d** — REFRESH. Highest impressions (517k) + 104 days stale — the clearest top-priority combo. Could be wrong if this traffic is the leftover of an old viral spike that's already cooled off.

**2. content_aaef01a50def** — REVIEW. Fresh content (22 days) already pulling huge traffic, so it's already doing well. Could be wrong if review turns up nothing to improve — the page may already be near-optimal.

**3. content_8c19996aa890** — REVIEW. 20 days old, very high traffic — flagging this early feels premature. Wrong if this is just launch-week traffic that naturally settles without any action.

**4. content_2cb567c3c89b** — REVIEW. 48 days, still in the fresh range, high visibility. Wrong if the traffic is seasonal and will decline on its own regardless of intervention.

**5. content_4c36c775b818** — REVIEW. Same pattern as rows 2-3 — 20 days, high impressions. Risk: the repeated identical `days_since_last_update` values look suspicious, possibly a data-generation artifact rather than a real signal.

**6. content_2dba2b1f9536** — REFRESH. Same 104-day-stale + high-impressions combo. Wrong if the content itself is fine and only a metadata timestamp is old, not the actual content.

**7. content_1a9e894be2e2** — REVIEW. 22 days old, strong traffic. Could be wrong if this is a specific client's outlier (e.g. a major brand that naturally gets high traffic with no real decline risk).

**8. content_2c2606c5d176** — REFRESH. Same stale+high pattern. Wrong if refreshing wouldn't actually move the needle — some content types (evergreen reference pages) stay stable without updates.

**9. content_db5989a78dd3** — REVIEW. Fresh + high traffic. Weak spot: if `days_since_last_update = 20` repeats across many rows, it might be a batch-upload date rather than actual content freshness.

**10. content_44e481c8f55b** — REVIEW. Same pattern again. Could be wrong if the rule is purely volume-driven and missing an actual content-quality issue.

**11. content_cb112fce36be** — REFRESH. Stale + high visibility. Wrong if this is "reference/evergreen" content that doesn't need updating (definitions, historical data).

**12. content_9532f197bbc8** — REFRESH. Same 104-day-stale cluster. Risk: if many stale rows share the exact same `days_since_last_update=104`, it may be a bulk-content-load date, not real staleness.

**13. content_36ff89c8214e** — REFRESH. High impressions, stale — priority looks reasonable. Wrong if the impression count itself is a stale/cached number that doesn't reflect current traffic.

**14. content_8e7ba84a972b** — REVIEW. Fresh, high traffic. Could be wrong if this content was already reviewed recently and flagging it again is redundant effort.

**15. content_b28d1efd668f** — REFRESH. Stale + visible. Wrong if the decline is unrelated to staleness — e.g. a competitor published better content, in which case a refresh alone might not be enough.

**16. content_89e84d699e9e** — REVIEW. Fresh + high impressions, part of the same cluster as above. Weak point: a purely volume-based ranking misses any content-quality signal.

**17. content_8451fc6f034d** — REVIEW. Same fresh-high-traffic group. Could be wrong if the "review" action has no clear next step — telling someone to "review" is vague without defining what to look for.

**18. content_aa4baf490b43** — REVIEW. Fresh, high impressions. Risk: if the rule is locked to "fresh+high=REVIEW", genuinely different underlying problems all get the same generic action.

**19. content_008fb02c46cb** — REVIEW. Same pattern. Wrong if this content's real issue is conversion/engagement, not visibility — REVIEW is too generic a label for that case.

**20. content_813e88069237** — REFRESH. Stale + high impressions, back to top-tier priority. Wrong if staleness (which our own signal check rated MIXED) turns out to be irrelevant here too — high impressions alone may be the real driver, staleness just coincidental.

**Note:** `days_since_last_update` only takes 3 distinct values (104, 22, 20) across this entire top-20 — unusually repetitive for a real dataset. Worth flagging in Section 4 (weak picks) as a possible data artifact rather than a true staleness signal.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [72]:
check["days_since_last_update"].value_counts().head(10)

,count
days_since_last_update,
20,11573
104,8773
22,3564
8,1929
13,515
14,505
28,466
25,441
7,420


In [73]:
leak_check_cols = ["trend_direction", "trend_pct", "is_declining"]
score_formula_cols = ["impressions_90d", "days_since_last_update"]

print("Columns used in rule:", score_formula_cols)
print("Leakage-risk columns (should NOT be in rule):", leak_check_cols)

Columns used in rule: ['impressions_90d', 'days_since_last_update']
Leakage-risk columns (should NOT be in rule): ['trend_direction', 'trend_pct', 'is_declining']


## Weak Picks

Across the full 30,000-row dataset, `days_since_last_update` is dominated by just three values: 20 (11,573 rows, ~38.6%), 104 (8,773 rows, ~29.2%), and 22 (3,564 rows, ~11.9%) — together nearly 80% of all rows. Real staleness data doesn't cluster this tightly around exact values; this pattern looks like a data-generation or batch-update artifact rather than genuine content freshness.

This directly weakens the `HIGH_VISIBILITY_STALE` reason code: pages aren't really "104 days stale" in any meaningful sense — they likely just share a batch timestamp. The staleness half of the rule was already rated MIXED in Signal Check #1, and this clustering is a plausible reason why — the signal may be measuring a data artifact rather than true content age. `impressions_90d` (the confirmed, CONFIRMED-rated signal) doesn't show this kind of clustering, so it remains the more trustworthy half of the rule.

## Leakage Check

The score, reason_code, and action columns were built only from `impressions_90d` and `days_since_last_update` — both available at scoring time, with no dependency on outcomes.

`trend_direction` and `trend_pct` were never used in the rule itself. They were only used in Section 1 to build a temporary `is_declining` column, solely to verify whether the staleness and visibility signals held — that column was for verification only and does not appear in the final score, reason_code, or action logic. No future-window or label-derived inputs were used to build the rule.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.